# Install Libraries

In [4]:
# Cài đặt các thư viện cho Model và API
!pip install -q transformers torch accelerate
!pip install -q fastapi uvicorn nest-asyncio pydantic

# Import Libraries

In [5]:
# Import thư viện xử lý bất đồng bộ và API
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import torch
# Import thư viện AI
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
# Import thư viện hỗ trợ chạy tunneling ngầm
import subprocess
import threading
import time

# Build Model

In [6]:
model_id = "Qwen/Qwen2.5-3B-Instruct"

print("Đang tải Tokenizer và Model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
)
print("Tải thành công!")

Đang tải Tokenizer và Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Tải thành công!


# Initialize API

In [7]:
app = FastAPI(title="Qwen LLM API")

# Lớp định nghĩa dữ liệu đầu vào
class QueryRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 256
    temperature: float = 0.7

@app.get("/")
def detail():
    return {"detail": "This is text generation API service using Qwen-2.5-3B LLM model"}

@app.get("/health")
def health_check():
    return {"status": "Qwen API is running on Colab"}

@app.post("/generate")
def generate_text(req: QueryRequest):
    # Chuẩn bị đầu vào cho model
    inputs = tokenizer(req.prompt, return_tensors="pt").to(model.device)

    # Sinh văn bản
    outputs = model.generate(
        **inputs,
        max_new_tokens=req.max_new_tokens,
        temperature=req.temperature,
        do_sample=True
    )

    # Giải mã kết quả (bỏ phần prompt ban đầu đi nếu muốn)
    response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"response": response_text}

In [8]:
# Cho phép uvicorn chạy ngầm trong Colab cell
nest_asyncio.apply()

# Hàm khởi động server
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

# Khởi động Server
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Mô phỏng Server khởi động
time.sleep(3)
print("Server đã chạy tại http://127.0.0.1:8000")

Server đã chạy tại http://127.0.0.1:8000


# Call Local API

In [9]:
import requests
# API
API_URL = "http://127.0.0.1:8000/"
#
parameters = {"prompt": "Thầy Lê Đức K ở trường Đại học Khoa học Tự nhiên có đẹp trai không?", "max_length": 512}
# Kiểm tra 'post' method
response = requests.post(API_URL + "generate", json=parameters)
print(response.json())
# Kiểm tra 'get' method
response2 = requests.get(API_URL)
print(response2.json())
# kiểm tra 'get' health method
response3 = requests.get(API_URL + "health")
print(response3.json())


{'response': 'Thầy Lê Đức K ở trường Đại học Khoa học Tự nhiên có đẹp trai không? - Tin tức\nNgày 23/06/2017 15:55 PM (GMT+7)\nSau khi được học sinh đánh giá cao, thầy Lê Đức K trở thành "hotboy" của trường.\nTrước đó, trên trang cá nhân, thầy Lê Đức K đã chia sẻ hình ảnh đang đi học với chiếc áo vest lịch lãm. Thầy còn đăng kèm dòng trạng thái: "Đi học luôn nhé, trời lạnh quá".\nNhững hình ảnh này nhanh chóng thu hút sự chú ý của cộng đồng mạng. Nhiều người tỏ ra thích thú và khen ngợi thầy Lê Đức K là một "hotboy" của trường.\nThầy Lê Đức K từng là giảng viên dạy môn hóa học tại Trường Đại học Khoa học Tự nhiên (ĐHQG TP.HCM). Hiện nay, thầy đã chuyển sang làm việc tại Trường Đại học Quốc tế Hồng Bàng.\nMột số hình ảnh thầy Lê Đức K chia sẻ trên trang cá nhân:\nThầy Lê Đức K trong bộ vest lịch lãm:\nNhiều người cho rằng, thầy Lê Đức K có vẻ ngoài điển trai và thu hút:\nCó người còn so sánh'}
{'detail': 'This is text generation API service using Qwen-2.5-3B LLM model'}
{'status': 'Qwen

# Call Public API

In [10]:
API_URL = "https://aqtmg-34-16-215-174.run.pinggy-free.link"
if API_URL:
    print(f"Thực hiện gọi API qua Pinggy: {API_URL}/generate")

    payload_tunnel = {
        "prompt": "Thủ đô của Việt Nam là gì?",
        "max_new_tokens": 512
    }

    # kiểm tra 'get' và 'post' method
    try:
        responsePost = requests.post(f"{API_URL}/generate", json=payload_tunnel)
        responseGet = requests.get(f"{API_URL}/")
        responseGetHealth = requests.get(f"{API_URL}/health")

        print("\nKết quả 'post' method:\n", responsePost.json().get("response", responsePost.text))
        print("\n Kết quả 'get' method:\n", responseGet.json().get("detail", responseGet.text))
        print("\n Kết quả 'get' health method:\n", responseGetHealth.json().get("status", responseGetHealth.text))
    except Exception as e:
        print("Lỗi khi gọi qua tunnel:", str(e))
else:
    print("Chưa có URL Pinggy để test.")

Thực hiện gọi API qua Pinggy: https://aqtmg-34-16-215-174.run.pinggy-free.link/generate

Kết quả 'post' method:
 Thủ đô của Việt Nam là gì? - VnExpress
Thứ tư, 14/10/2015, 10:37 (GMT+7)
Thủ đô của Việt Nam là Hà Nội / Hanoi.
Hà Nội là thành phố lớn nhất và trung tâm chính trị, kinh tế, văn hóa, giáo dục, khoa học công nghệ và giao thông vận tải của Việt Nam. Thành phố này nằm ở phía Bắc Việt Nam, cách Hà Nội khoảng 100km về phía Tây Nam. Hà Nội có diện tích 2.181,4 km2 với dân số khoảng 2,5 triệu người. Hà Nội được gọi là "Thủ đô ngàn năm văn hiến" bởi lịch sử lâu đời, với nhiều di tích lịch sử, văn hóa, kiến trúc độc đáo. Từ thế kỷ thứ 10, Hà Nội đã trở thành trung tâm chính trị, kinh tế và văn hóa của nước Đại Việt. Năm 1976, Hà Nội trở thành thủ đô mới của nước Việt Nam Dân chủ Cộng hòa. Sau khi đất nước thống nhất, Hà Nội tiếp tục giữ vai trò trung tâm chính trị, kinh tế, văn hóa, giáo dục, khoa học công nghệ và giao thông vận tải của cả nước. Hà Nội cũng là thành phố đầu tiên của 